# 07 — Reference App Walkthrough

You have seen each library on its own — engines, inspection, memory, retrieval.
Composition is a different problem. Memory backends and engines fight over lifecycle.
Packages disagree on event shapes. An error in one component cascades through the next.
A working library is not the same as a working application.

`language_tutor` is this course's answer — a real application built on the libraries
from notebooks 02–06. It runs on your machine, uses your hardware, and exposes its
internals through the same `llm_harness_core` contracts every other package speaks.
This notebook walks the assembly: how the pieces fit, what the data looks like at each
seam, and where the extension points live.

**Prerequisites:** notebooks 02, 03, 04. Most cells in this notebook use a `FakeEngine`
and do not require Ollama. The optional live-model section at the end does.


## The composition problem

You can write a working memory system. You can write a working engine abstraction.
Combining them is harder than it looks.

Here is what goes wrong in a naive assembly:

- **Session lifecycle.** Memory backends need to be opened and closed. Engines consume VRAM
  and should be unloaded when not in use. Who is responsible for that lifecycle?
- **Backend selection.** The app needs to work with `engram_lite` for teaching and full `engram`
  for richer behavior. If the session code calls the backend directly, swapping is a large change.
- **Contract boundaries.** The inspector, the workbench, and future tools need to *observe*
  what happened in a turn. If each package emits its own event shape, nothing composes.
- **Error propagation.** When the engine is unreachable, the app should degrade cleanly.
  When memory is empty, the first turn should still work.

The `language_tutor` codebase is the answer to each of these. Reading it shows you
the actual decisions, not just the goal.


## The stack map

```
language_tutor
├── llm_engines          engine selection and backend access
├── engram_lite          default lightweight memory path
├── engram               optional advanced memory path
├── llm_harness_core     shared capability/result/trace contracts
├── llm_inspector        intended comparison and reporting path
└── llm_inspector_ui     intended visual workbench path
```

The `reference-stack` endpoint on the app returns this map as a machine-readable object.
You can call it without starting a server:


In [ ]:
import asyncio
from language_tutor.routes.session import get_reference_stack

stack = asyncio.run(get_reference_stack(language='spanish', memory_backend='engram_lite'))

print(f'App:            {stack.app}')
print(f'Learning stage: {stack.learning_stage}')
print(f'Memory backend: {stack.memory_backend}')
print(f'Required:       {stack.required_packages}')
print(f'Optional:       {stack.optional_packages}')
print(f'Observability:  {stack.observability_packages}')
print()
print('Current paths:')
for k, v in stack.current_paths.items():
    print(f'  {k}: {v}')


The `capability` field carries a `CapabilityDescriptor` from `llm_harness_core` —
the same type emitted by engines, augmenters, and memory backends:


In [ ]:
import pprint
pprint.pprint(stack.capability)


## Setup: running without a live model

Most cells in this notebook run without Ollama. We replace the engine with a
`FakeEngine` — the same approach used in the test suite.

The `FakeEngine` implements exactly the interface `LLMEnginesAdapter` expects:
`generate()`, `generate_structured()`, and `count_tokens()`.
Patching `EngineManager._load_engine` at class level ensures the fake engine
is used for both the executor (loaded at construction) and the planner (loaded lazily in `start()`).


In [ ]:
import asyncio
import copy
import json
import tempfile
from pathlib import Path
from unittest.mock import patch

from pydantic import BaseModel

from language_tutor.engine_manager import EngineManager
from language_tutor.hardware_strategy import STRATEGIES
from language_tutor.tutor_session import TutorSession


class _Plan(BaseModel):
    warmup_topic: str = 'daily life'
    focus_areas: list[str] = ['past tense', 'ser vs estar']
    drill_type: str = 'mixed_review'
    new_content: list[str] = ['mercado', 'ayer']
    estimated_minutes: dict[str, int] = {'warmup': 5, 'conversation': 7, 'drill': 3}


class FakeEngine:
    """Deterministic stand-in for any llm_engines backend."""
    model_name = 'fake-tutor-model'

    def generate(self, prompt: str, **kwargs) -> str:
        if 'summary' in prompt.lower():
            return 'Session summary: practiced past tense and ser/estar.'
        return 'Claro. Ayer fui al mercado y compré frutas frescas. ¿Y tú?'

    def generate_structured(self, prompt: str, response_model, **kwargs):
        return response_model(**_Plan().model_dump())

    def count_tokens(self, text: str) -> int:
        return max(1, len(text.split()))


def build_session(base_dir: Path, session_id: str = 'nb07', memory_backend: str = 'engram_lite') -> TutorSession:
    """Build a TutorSession with FakeEngine — no Ollama required."""
    strategy = copy.deepcopy(STRATEGIES['local_everything'])
    return TutorSession(
        language='spanish',
        strategy=strategy,
        base_dir=base_dir,
        session_id=session_id,
        memory_backend=memory_backend,
    )


# Patch EngineManager so every _load_engine call returns FakeEngine.
# This context manager is re-opened in each section that builds a session.
_engine_patch = patch.object(EngineManager, '_load_engine', lambda self, cfg, purpose: FakeEngine())

print('Setup complete. FakeEngine is ready.')


## Follow one turn through the stack

This is the most important exercise in Stage 5.
A single user message travels through five steps before the response reaches the caller.
Reading the code is valuable; seeing the data at each step is more so.


In [ ]:
# Step 0: Build and start the session
demo_dir = Path(tempfile.mkdtemp())

with _engine_patch:
    session = build_session(demo_dir, session_id='nb07_turn')
    start_result = asyncio.run(session.start(duration_minutes=20))

print('Session state:', session.state.value)
print('Greeting:', start_result['greeting'])
print('Plan focus areas:', start_result['plan'].get('focus_areas', []))


### Step 1: User message → working memory

`handle_text()` starts by calling `self.memory.add_turn('user', user_input)`.
This adds the turn to working memory *before* generating a response — so the model
sees the current turn as part of the conversation history, not just as the current prompt.


In [ ]:
user_message = 'Ayer fui al mercado y compré pan fresco.'

# Step 1: what does build_prompt return?
with _engine_patch:
    session.memory.add_turn('user', user_message)
    prompt_result = session.memory.build_prompt(user_message=user_message)

print('Keys returned by build_prompt():')
for k, v in prompt_result.items():
    if k == 'prompt':
        print(f'  {k}: [{len(v)} chars] ...')
    else:
        print(f'  {k}: {v}')


The prompt returned by `build_prompt()` is what the executor actually receives.
It has already had working memory, episodic context, and semantic memory injected
according to the token budget. `prompt_tokens` tells you how much budget was used;
`compressed` tells you whether older turns were summarised to fit.

### Step 2: Enriched prompt → executor

The executor gets the enriched prompt plus the language-specific system prompt.
The system prompt is stored on the `LanguageProfile`, not on the memory engine.


In [ ]:
# Step 2: what does the executor see?
print('System prompt (first 200 chars):')
print(session.profile.system_prompt[:200])
print()

# Step 3: run the full turn
with _engine_patch:
    response = asyncio.run(session.handle_text(user_message))

print('Response text:', response.text)
print('Corrections:', response.corrections)
print('New vocabulary:', response.new_vocabulary)
print('Metadata:', response.metadata)


### Step 3: Response → working memory + semantic extraction

After the executor returns, `handle_text()` stores the assistant turn back into working memory,
runs lightweight correction and vocabulary extraction, and updates semantic memory.
By the time the caller receives the `TutorResponse`, memory already reflects this turn.


In [ ]:
# What is in working memory after one turn?
with _engine_patch:
    turns = session.memory.get_recent_turns(n=10)

print(f'Working memory has {len(turns)} turns:')
for t in turns:
    print(f'  [{t.role}] {t.content[:80]}')

# Memory stats after one exchange
with _engine_patch:
    stats = session.memory.get_stats()

print()
print('Memory stats:')
print(f'  working.message_count: {stats["working"]["message_count"]}')
print(f'  episodic.count:        {stats["episodic"]["count"]}')
print(f'  backend:               {stats["backend"]}')


In [ ]:
session.close()


## Compare memory backends

The `TutorMemoryBackend` protocol means the session code is identical
regardless of which backend is active. The backends differ in:

- **Persistence format.** `engram_lite` uses SQLite + optional ChromaDB.
  `engram` uses a multi-layer runtime with daemon extraction.
- **Extraction depth.** `engram_lite` uses lightweight ingestion scoring.
  `engram` adds a Reflex regex layer and an optional Cognitive LLM layer.
- **Startup cost.** `engram_lite` is near-instant. `engram` initialises
  additional layers including the neural memory config.

Run the same three turns through both backends and compare their stats:


In [ ]:
TURNS = [
    'Hoy quiero practicar el pretérito. ¿Puedes ayudarme?',
    'Ayer yo fui al mercado y compré verduras frescas.',
    'Mi hermana estudiaba medicina pero ahora trabaja en un hospital.',
]


def run_session(memory_backend: str, base: Path) -> dict:
    with _engine_patch:
        s = build_session(base, session_id=f'compare_{memory_backend}', memory_backend=memory_backend)
        asyncio.run(s.start(duration_minutes=15))
        for turn in TURNS:
            asyncio.run(s.handle_text(turn))
        stats = s.memory.get_stats()
        s.close()
    return stats


base_lite = Path(tempfile.mkdtemp())
base_engram = Path(tempfile.mkdtemp())

stats_lite   = run_session('engram_lite', base_lite)
stats_engram = run_session('engram',      base_engram)

print('After 3 turns + planning turn:')
print()
print(f'{"Field":<30} {"engram_lite":<20} {"engram"}')
print('-' * 65)

def _s(d, *keys):
    v = d
    for k in keys:
        v = v.get(k, 'N/A') if isinstance(v, dict) else 'N/A'
    return str(v)

rows = [
    ('backend',                'backend'),
    ('working.message_count',  ('working', 'message_count')),
    ('episodic.count',         ('episodic', 'count')),
    ('semantic.available',     ('semantic', 'available')),
    ('vector_search.available',('vector_search', 'available')),
    ('config.total_tokens',    ('config', 'total_prompt_tokens')),
]

for label, key_path in rows:
    keys = key_path if isinstance(key_path, tuple) else (key_path,)
    lite_val   = _s(stats_lite, *keys)
    engram_val = _s(stats_engram, *keys)
    print(f'{label:<30} {lite_val:<20} {engram_val}')


The working memory counts may differ because `engram` runs background extraction
that can store episodic summaries asynchronously. `engram_lite` is synchronous
and deterministic — every turn that passes the importance threshold is stored immediately.

The `TutorMemoryBackend` protocol ensures `handle_text()` never needs to know which
backend is active:


In [ ]:
from language_tutor.memory_backend import TutorMemoryBackend
import inspect

# The protocol defines the boundary both backends must satisfy
methods = [
    name for name, _ in inspect.getmembers(TutorMemoryBackend, predicate=inspect.isfunction)
    if not name.startswith('_')
]
print('TutorMemoryBackend protocol methods:')
for m in methods:
    sig = inspect.signature(getattr(TutorMemoryBackend, m))
    print(f'  .{m}{sig}')


## The interop layer

`TutorSession` exposes two sets of methods for every operation:

| Plain method | Interop method |
|---|---|
| `start()` | `start_interop()` |
| `handle_text()` | `handle_text_interop()` |
| `explain()` | `explain_interop()` |

The plain methods return domain objects (`TutorResponse`, `dict`).
The interop methods return `OperationResult` from `llm_harness_core` — the same
contract type used by `llm_inspector_ui`'s augmenter service.

The interop layer exists so that observation tools (inspectors, workbenches, evaluation
harnesses) can consume tutor output without importing `language_tutor` directly.


In [ ]:
from llm_harness_core import CapabilityDescriptor, MemoryRecord, OperationResult, TraceEvent

base_interop = Path(tempfile.mkdtemp())

with _engine_patch:
    isession = build_session(base_interop, session_id='nb07_interop')
    start_result = asyncio.run(isession.start_interop(duration_minutes=15))

print('Type:', type(start_result).__name__)
print('ok:', start_result.ok)
print('value keys:', list(start_result.value.keys()))
print('diagnostics keys:', list(start_result.diagnostics.keys()))


In [ ]:
# Capability descriptor — describes what this session can do
cap = start_result.diagnostics['capability']
print('CapabilityDescriptor:')
print(f'  provider:   {cap.provider}')
print(f'  component:  {cap.component}')
print(f'  features:   {list(cap.features)}')
print()

# Trace events — what happened during this operation
events = start_result.diagnostics['trace_events']
print(f'TraceEvents ({len(events)}):')
for e in events:
    print(f'  [{e.severity}] {e.event_type}')
    print(f'     source: {e.source_package}.{e.source_component}')
    if e.message:
        print(f'     message: {e.message}')


In [ ]:
# Memory records — recent turns as shared interop objects
records = start_result.diagnostics['memory_records']
print(f'MemoryRecords ({len(records)}):')
for r in records:
    role = r.metadata.get('role', '?')
    print(f'  [{role}] {r.text[:60]}')

print()

# Run one turn and inspect the interop result
with _engine_patch:
    turn_result = asyncio.run(isession.handle_text_interop('Hola, estoy listo para practicar.'))

print('handle_text_interop() result:')
print('  value keys:', list(turn_result.value.keys()))
print('  response text:', turn_result.value['text'][:60])
print('  trace event:', turn_result.diagnostics['trace_events'][0].event_type)


The `OperationResult` is what the inspector and workbench consume.
See the exercises at the end for how to feed it to `InspectorService`.


In [ ]:
isession.close()


## Engine swapping

The strategy dict controls which engine handles planning and which handles execution.
Changing backends is a one-value edit:


In [ ]:
# Inspect the local_everything strategy
strategy = STRATEGIES['local_everything']
print('local_everything:')
print(f'  planner:  {strategy["planner"]["engine"]} / {strategy["planner"]["model"]}')
print(f'  executor: {strategy["executor"]["engine"]} / {strategy["executor"]["model"]}')
print(f'  cost:     ${strategy["cost_per_session"]}/session')
print()

# Inspect a cloud-hybrid strategy
hybrid = STRATEGIES['hybrid_cloud_planning']
print('hybrid_cloud_planning:')
print(f'  planner:  {hybrid["planner"]["engine"]} / {hybrid["planner"]["model"]}')
print(f'  executor: {hybrid["executor"]["engine"]} / {hybrid["executor"]["model"]}')
print(f'  cost:     ${hybrid["cost_per_session"]}/session')


The `TutorSession` constructor gets the strategy dict and calls
`EngineManager._load_engine()` for each role. The session code never
checks `engine_type` directly — that decision is encapsulated in
`LLMEnginesAdapter.build_engine()`, which maps the strategy dict to the
correct `llm_engines` factory call.

```
strategy['planner']['engine'] == 'anthropic'
    → build_engine({'engine': 'anthropic', 'model': 'claude-opus-4-7', ...})
        → EngineFactory.create('anthropic', model=..., api_key=...)
            → LLMEnginesAdapter(engine)
```

The session calls `adapter.generate()` — same call regardless of whether
the backend is Ollama, Anthropic, or the FakeEngine used in this notebook.

**Live Ollama path** (requires Ollama running):


In [ ]:
import os

if os.getenv('LANGUAGE_TUTOR_LIVE_OLLAMA'):
    # No FakeEngine patch — use real Ollama via the default strategy
    live_dir = Path(tempfile.mkdtemp())
    live = TutorSession(
        language='spanish',
        strategy=STRATEGIES['local_everything'],
        base_dir=live_dir,
        session_id='nb07_live',
    )
    start_result = asyncio.run(live.start(duration_minutes=10))
    response = asyncio.run(live.handle_text('Hola. Quiero practicar el pretérito.'))
    print('Live response:', response.text)
    live.close()
else:
    print('Set LANGUAGE_TUTOR_LIVE_OLLAMA=1 to run with a real Ollama model.')
    print('This cell is skipped in the default path.')


## Running it for real

Everything above uses `FakeEngine` so cells run instantly without touching Ollama.
That's the right setup for a notebook walkthrough, but you should run the actual
application at least once to see the pieces work together.

The app launches from a shell script in the package root:

```bash
cd ai_tools/language_tutor
./start.sh                  # web UI on port 8080
./start.sh --setup          # interactive setup wizard (pick a strategy)
./start.sh --preflight      # check that engines, models, and keys are reachable
./start.sh --cli            # command-line interactive session, no UI
./start.sh --test           # quick smoke test, no UI
```

What you'll see on first run:

1. **Setup wizard** asks which strategy fits your hardware. The strategies you
   explored two cells above are exactly the choices it offers.
2. **Preflight check** verifies that Ollama is reachable (or that the relevant
   API key is set), the chosen models are pulled, and the memory backend opens
   cleanly. The output mirrors `stack.capability` from earlier in this notebook —
   same data, different presentation.
3. **The web UI** opens at `http://localhost:8080`. The text chat works
   immediately. Voice mode needs microphone access, which most browsers only
   grant over HTTPS; if you want voice, generate a self-signed cert and serve
   on `https://localhost:8443` per the notes in `language_tutor/README.md`.

Once it's running, open `http://localhost:8080/reference-stack?language=spanish`
in another tab. You'll see the same `ReferenceStack` object you constructed at
the top of this notebook, served live by the app. The endpoint exists so external
tools — the inspector, an evaluation harness, your own scripts — can introspect
what's wired without importing `language_tutor`.

A few things worth doing in the live app:

- Send a Spanish sentence with a deliberate error. Watch how the response
  flags it. The correction logic is in `tutor_session.handle_text()`; this is
  the same path you traced through the FakeEngine earlier.
- Open the memory inspector tab. The contents are the `MemoryRecord` objects
  you saw printed in the interop section.
- Switch strategies in the setup wizard mid-session. The session restarts but
  the memory persists — backends are storage, not state.


## Extension points

Two integrations are visible in the code but not yet wired. Both are useful exercises
once you finish the course path.

### Where `rag_lib` fits

`rag_lib` is not a current dependency of `language_tutor`, but the integration point
is concrete: the planner's prompt-building method. Right now the planner reads
working memory and a small set of static profile fields. With `rag_lib`, lesson
content could be grounded in a curated corpus — a grammar reference, a textbook,
the user's own marked-up notes.

The wiring would look like this. Imagine you have a curriculum corpus already
ingested with the pipeline you built in notebook 05:

```python
from rag_lib import RAGPipeline

# Ingested once, lives at ~/.rag_lib/chroma or wherever your config points
curriculum = RAGPipeline(config="~/.rag_lib/curriculum.yaml")

# Inside TutorSession, alongside the existing _build_planning_prompt():
def _build_planning_prompt_with_rag(self, user_message: str) -> str:
    # Existing prompt assembly...
    base_prompt = self._build_planning_prompt(user_message)

    # NEW: retrieve curriculum-grounded context for the planner
    focus = self._infer_lesson_focus(user_message)  # e.g. "Spanish preterite"
    chunks = curriculum.retrieve(focus, n_results=3)
    grounding = curriculum.assemble_prompt(focus, chunks, max_context_tokens=800)

    return f"{base_prompt}\n\n## Reference material\n{grounding}"
```

Two things to notice. First, `RAGPipeline` returns chunks the same shape regardless
of what's stored in it, so the integration is symmetric — the tutor doesn't have to
know whether the corpus is a textbook, a grammar reference, or the user's notes.
Second, the planner is the right place for retrieval. The executor runs per turn
and budget matters; the planner runs once per session and can afford a richer prompt.

A nice exercise for after notebook 06: ingest a grammar reference into `rag_lib`,
add the call above to a fork of `tutor_session.py`, and compare the lesson plans
the planner produces with and without grounding.

### Where `llm_inspector_ui` fits

The tutor already emits `OperationResult` with `TraceEvent` and `MemoryRecord`
objects. `InspectorService.inspect()` in `llm_inspector_ui` can normalise these
into a trace dict. The missing piece is a dedicated normaliser for `OperationResult`
(as opposed to `AugmentResult`) — that normaliser would map tutor trace events to
prompt sections the workbench can render.

The gap is tracked in the repo's `STATUS.md` under the `llm_inspector_ui` heading.


## Learning checkpoint

After completing this notebook, you should be able to answer:

1. Why does `handle_text()` add the user turn to memory *before* calling the executor?
2. What is the difference between `handle_text()` and `handle_text_interop()`,
   and when would you use each?
3. If you wanted to add a third memory backend (e.g., a Redis-backed store),
   what interface would it need to implement?
4. What would break if `TutorSession` called `OllamaEngine` directly instead
   of routing through `EngineManager` → `LLMEnginesAdapter`?
5. Where exactly in the codebase would you add a `rag_lib` retrieval call
   to ground the session plan in a curated grammar reference?


## Exercises

**1. Observe memory compression.**
Run 15 turns through a session with a tight token budget:
```python
result = session.memory.build_prompt(user_message='test', max_prompt_tokens=200)
print(result['compressed'], result['prompt_tokens'])
```
At what turn count does `compressed` become `True`? What changes in the prompt?

**2. Swap memory backends mid-comparison.**
Run five identical turns through `engram_lite` and `engram` (use `run_session()`
from this notebook). Print `stats['episodic']['count']` for both.
What differs? Look at `stats['episodic']['quality']` for the explanation.

**3. Inspect with InspectorService.**
Take the `turn_result` from the interop section and feed it to the inspector:
```python
from llm_inspector_ui.services.inspector_service import InspectorService
from llm_inspector_ui.utils.trace_access import get_token_accounting
inspector = InspectorService()
trace = inspector.inspect(
    augmenter_id='language_tutor',
    augment_result=turn_result,
    user_text='your message here',
    session_id=isession.session_id,
)
print(get_token_accounting(trace))
```
What does the trace contain? Why are `sections` empty? (See the inline comment
in `test_tutor_inspector_spine.py` for the explanation.)

**4. Add a custom system prompt.**
After `build_session()`, override `session.profile.system_prompt` with a
more directive prompt. Run a turn and observe whether the correction style changes.
Does `engram_lite` pick up the new prompt automatically, or is it fixed at init?

**5. Implement a minimal third backend.**
Write a class `InMemoryTutorMemory` that satisfies `TutorMemoryBackend`
using only Python lists — no files, no SQLite.
Pass it to `TutorSession` by monkey-patching `session.memory` after construction.
Run three turns. What does not work? Why?
